In [15]:
# multi class classification of FGL sources using tree categories
import numpy as np
import pandas as pd
from matplotlib import pyplot
#from scipy.stats import binom
print(np.__version__)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.calibration import calibration_curve

import time
import os
import sys
if '../lib/' not in sys.path:
    sys.path.append('../lib/')
import dio3
import multi_fgl_utils as util
import plot_util as putil

1.24.3


In [16]:
if 1:
    import importlib
    importlib.reload(util);


In [17]:
# load the cat tree
test_frac = 0.3
weighted = False
max_sample = 7

folder = '../data_out/'
tree_alg = 'GMM' # 'stat', 'GMM', 'RF', '4class'
mode = 'all_assoc' # 'all_assoc', 'no_bcu_spp'
if mode == 'all_assoc':
    mode_str = ''
else:
    mode_str = '_%s' % mode
feature_mode = 'all features' # 'all features', 'no coord features'

if tree_alg == 'GMM':
    nmin = 100
    max_depth = 4
    tree_cat_fn = folder + 'tree_cat_nmin%i%s.yaml' % (nmin, mode_str)
if tree_alg == 'RF':
    max_depth = 4
    if mode == 'no_bcu_spp':
        nmin = 60
    else:
        nmin = 100
    tree_cat_fn = folder + 'tree_cat_nmin%i_RF%s.yaml' % (nmin, mode_str)
elif tree_alg == 'stat':
    tree_cat_fn = folder + 'tree_cat_stat.yaml'
elif tree_alg == 'RF_aakash':
    tree_cat_fn = folder + 'tree_cat_rf_15_50.yaml'
elif tree_alg == '4class':
    tree_cat_fn = folder + 'tree_cat_4class.yaml'
elif tree_alg == '2class':
    tree_cat_fn = folder + 'tree_cat_2class_gal_egal.yaml'
    title_dict = {'00': 'Extragalactic', '01': 'Galactic'}

tree_cat_dict = dio3.loaddict(tree_cat_fn)
if max_depth is not None:
    tree_cat_dict = util.trim2depth(tree_cat_dict, depth=max_depth)
nclasses = len(tree_cat_dict)
max_tree_cat_dict = util.tree2max_tree(tree_cat_dict)
dec_tree_cat_dict = util.max_tree2dec_tree(max_tree_cat_dict)
nodes = [str(key) for key in dec_tree_cat_dict.keys()]

alg_descr = '_%s' % tree_alg
if tree_alg == 'GMM' or tree_alg == 'RF':
    alg_descr += '_nmin%s' % (nmin)
elif tree_alg == '2class':
    alg_descr += '_gal_egal'
alg_descr += mode_str
if feature_mode == 'no coord features':
    alg_descr += '_no_coord_features'

Eind = 1000
xdf_fn = folder + 'xdf_fgl_Eind%iMeV.csv' % Eind
#xdf_fn = folder + 'xdf_fgl.csv'
xdf = pd.read_csv(xdf_fn, index_col=0)
all_features = [name for name in xdf.columns if name.find('CLASS') == -1]
if feature_mode == 'all features':
    features = all_features
elif feature_mode == 'no coord features':
    features = [feat for feat in all_features if feat.find('GL') == -1]
ns = len(xdf)


assoc_classes = []
for cls in tree_cat_dict.values():
    assoc_classes.extend(cls)
nsrc_dict = {cl:np.sum(xdf['CLASS1'] == cl) for cl in assoc_classes}

pars_dict = {'RF':{'max_depth': 15, 'n_estimators': 50,},
             'NN_SK':{'activation': 'tanh', 'hidden_layer_sizes': (20, 10,),},
             'BDT':{'max_depth': 15},
             'LR':{'max_iter':1000},
             'NN': {'optimizer': 'adam', 'learning_rate': 0.001, 'epochs': 1000, 'activation': 'tanh', 
                       'hidden_layer_sizes': [20, 10], 
                       'nfeatures': len(features), 'classes': nodes, 'verbose': 0,
                       'drop_rate': 0.0,
                      'validation_split': None}
            }

if weighted:
    pars_dict['NN']['epochs'] = 500

feature_str = feature_mode.replace(' ', '_')
weights_fn = '%s/weights/weights_%s.csv' % (folder, feature_str)
weights_df = pd.read_csv(weights_fn, index_col=0)
sample_ratio_0 = util.get_sample_ratio(weights_df['unas_pdf'], weights_df['assoc_pdf'], binary=False, 
                                       max_sample=max_sample)

if weighted:
    weight_str = '_weighted'
    sample_ratio = sample_ratio_0
else:
    sample_ratio = None
    weight_str = ''
    
folder_out = '../results/4FGL-DR3_probabilistic_classification_weighted/'
if not os.path.isdir(folder_out):
    os.mkdir(folder_out)

fn_out = folder_out + '4FGL-DR3_%iclass%s%s_prob_cat.csv' % (nclasses, alg_descr, weight_str)

#class_def_fn = folder_out + '4FGL-DR3_%iclass%s_groups_def.csv' % (nclasses, alg_descr)

algs = ['RF', 'NN']
#algs = ['RF',]
print(nclasses)
print(tree_cat_fn)
print(fn_out)
print(fn_out in os.listdir('../results/4FGL-DR3_probabilistic_classification/'))

6
../data_out/tree_cat_nmin100.yaml
../results/4FGL-DR3_probabilistic_classification_weighted/4FGL-DR3_6class_GMM_nmin100_prob_cat.csv
False


In [18]:
dec_tree_cat_dict, max_tree_cat_dict

({1: ['sbg', 'rdg', 'bll', 'css'],
  2: ['bcu', 'ssrq'],
  3: ['fsrq', 'sey', 'nlsy1', 'agn'],
  4: ['glc', 'nov', 'spp'],
  5: ['msp', 'lmb'],
  6: ['hmb', 'psr', 'sfr', 'snr', 'gc', 'pwn', 'gal', 'bin']},
 {'00000': ['sbg', 'rdg', 'bll', 'css'],
  '00100': ['bcu', 'ssrq'],
  '01000': ['fsrq', 'sey', 'nlsy1', 'agn'],
  '01100': ['glc', 'nov', 'spp'],
  '01101': ['msp', 'lmb'],
  '01110': ['hmb', 'psr', 'sfr', 'snr', 'gc', 'pwn', 'gal', 'bin']})

In [19]:
ys = np.zeros(ns, dtype='<U6')
clf_mask = np.zeros(ns, dtype=bool)
for key, cls in dec_tree_cat_dict.items():
    cls_mask = util.get_cl_mask(cls, xdf, cname='CLASS1')
    ys[cls_mask] = key
    clf_mask[cls_mask] = True
    
X = xdf[features][clf_mask]
X_unas = xdf[features][~clf_mask]
y = ys[clf_mask]


classif_columns = ['%s_%s' % (node, alg) for node in nodes for alg in algs]
print(classif_columns)



['1_RF', '1_NN', '2_RF', '2_NN', '3_RF', '3_NN', '4_RF', '4_NN', '5_RF', '5_NN', '6_RF', '6_NN']


In [20]:
add_sample_weights_only = True
if add_sample_weights_only:
    cat_df = pd.read_csv(fn_out, index_col=0)
    cat_df['sample_weight'] = sample_ratio_0
    cat_df['sample_weight'][~clf_mask] = 1.
    print('Save the prob catalog to file:')
    print(fn_out)
    print(cat_df.columns)
    cat_df.to_csv(fn_out)
cat_df[['CLASS1', 'sample_weight']]

Save the prob catalog to file:
../results/4FGL-DR3_probabilistic_classification_weighted/4FGL-DR3_6class_GMM_nmin100_prob_cat.csv
Index(['sin(GLAT)', 'cos(GLON)', 'sin(GLON)', 'log10(Energy_Flux100)',
       'log10(Unc_Energy_Flux100)', 'log10(Signif_Avg)', 'LP_index1000MeV',
       'LP_beta', 'LP_SigCurv', 'log10(Variability_Index)', 'CLASS1', '1_RF',
       '1_RF_std', '1_NN', '1_NN_std', '2_RF', '2_RF_std', '2_NN', '2_NN_std',
       '3_RF', '3_RF_std', '3_NN', '3_NN_std', '4_RF', '4_RF_std', '4_NN',
       '4_NN_std', '5_RF', '5_RF_std', '5_NN', '5_NN_std', '6_RF', '6_RF_std',
       '6_NN', '6_NN_std', 'sample_weight'],
      dtype='object')


/var/folders/mb/yxqk032n4lvgsvnl3lqzbwvr0000gn/T/ipykernel_41413/1628543263.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cat_df['sample_weight'][~clf_mask] = 1.


,CLASS1,sample_weight
4FGL J0000.3-7355,unas,1.000000
4FGL J0000.5+0743,unas,1.000000
4FGL J0000.7+2530,unas,1.000000
4FGL J0001.2+4741,bcu,0.691970
4FGL J0001.2-0747,bll,0.000002
...,...,...
4FGL J2359.3-2049,bll,0.000656
4FGL J2359.3+1444,unas,1.000000
4FGL J2359.7-5041,bcu,0.139090
4FGL J2359.9-3736,bcu,0.374524


In [21]:
y_pred = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0.)
y2_pred = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0.)
nn_df = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0)


In [22]:
low_lim = 5
for alg in algs:
    j = -1
    print(alg)
    alg_clms = [clm for clm in classif_columns if clm.find(alg) > -1]
    while np.min(nn_df[alg_clms].values) < low_lim:
        if alg.startswith('NN'):
            time0 = time.time()
        j += 1
        print(j, np.min(nn_df[alg_clms].values))
        clf = util.get_classifier(alg, pars_dict[alg])
        if 1:
            clf_inds = np.arange(len(xdf))[clf_mask]
            rstate = np.random.RandomState(seed=j)
            test_inds = rstate.choice(clf_inds, size=int(len(clf_inds) * test_frac), replace=False, p=None)
            train_inds = [i for i in clf_inds if i not in test_inds]
            
            # add weights to training samples
            if weighted:
                sample_ratio_train = sample_ratio.iloc[train_inds]
            else:
                sample_ratio_train = None
            # add weights to testing samples
            #pred_dict_all[j]['test_weights'] = sample_ratio.iloc[test_inds]

            X_train = xdf[features].iloc[train_inds]
            X_test = xdf[features].iloc[test_inds]
            y_train = ys[train_inds]
            y_test = ys[test_inds]
            
            #clf.fit(X_train, y_train, sample_weight=sample_ratio_train)
        else:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_frac, 
                                                                random_state=j)
        if alg == 'NN_SK':
            clf.fit(X_train, y_train)
        else:
            clf.fit(X_train, y_train, sample_weight=sample_ratio_train)
            
        pred_test = clf.predict_proba(X_test).T
        pred_unas = clf.predict_proba(X_unas).T

        class_inds = [list(clf.classes_).index(node) for node in nodes]
        for i, node in enumerate(nodes):
            clm = '%s_%s' % (node, alg)
            src_names = X_test.index
            y_pred[clm].loc[src_names] += pred_test[class_inds[i]]
            y2_pred[clm].loc[src_names] += pred_test[class_inds[i]]**2
            nn_df[clm].loc[src_names] += 1
            src_names = X_unas.index
            y_pred[clm].loc[src_names] += pred_unas[class_inds[i]]
            y2_pred[clm].loc[src_names] += pred_unas[class_inds[i]]**2
            nn_df[clm].loc[src_names] += 1
        if alg.startswith('NN'):
            print('Step time: %.2f s' %  (time.time() - time0))


RF
0 0
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 0
10 0
11 0
12 0
13 0
14 0
15 0
16 0
17 0
18 0
19 0
20 0
21 0
22 0
23 0
24 1
25 1
26 1
27 1
28 2
29 2
30 2
31 2
32 2
33 2
34 2
35 3
36 3
37 3
38 3
39 3
40 3
41 3
42 4
43 4
44 4
NN
0 0
Step time: 31.73 s
1 0
Step time: 29.49 s
2 0
Step time: 34.22 s
3 0
Step time: 36.67 s
4 0
Step time: 32.90 s
5 0
Step time: 34.59 s
6 0
Step time: 43.51 s
7 0
Step time: 42.44 s
8 0
Step time: 46.25 s
9 0
Step time: 40.47 s
10 0
Step time: 46.24 s
11 0
Step time: 49.25 s
12 0
Step time: 41.58 s
13 0
Step time: 41.83 s
14 0
Step time: 42.41 s
15 0
Step time: 44.61 s
16 0
Step time: 44.98 s
17 0
Step time: 44.81 s
18 0
Step time: 53.98 s
19 0
Step time: 43.96 s
20 0
Step time: 53.68 s
21 0
Step time: 44.98 s
22 0
Step time: 51.15 s
23 0
Step time: 54.77 s
24 1
Step time: 51.05 s
25 1
Step time: 52.07 s
26 1
Step time: 44.99 s
27 1
Step time: 45.88 s
28 2
Step time: 45.03 s
29 2
Step time: 56.07 s
30 2
Step time: 84.90 s
31 2
Step time: 51.91 s
32 2
Step time: 49.41

In [23]:
y_pred_norm = y_pred / nn_df
y_std = np.sqrt((y2_pred - y_pred_norm**2 * nn_df) / (nn_df - 1.))

In [24]:
#sdf.index = range(1, len(sdf) + 1)

In [25]:
# output the catalog
df_out = xdf.copy()
for clm in classif_columns:
    df_out[clm] = y_pred_norm[clm]
    df_out[clm + '_std'] = y_std[clm]
df_out['sample_weight'] = sample_ratio_0
df_out['sample_weight'][~clf_mask] = 1. # put 1 for unassociated sources

print('Save the prob catalog to file:')
print(fn_out)
df_out.to_csv(fn_out)

Save the prob catalog to file:
../results/4FGL-DR3_probabilistic_classification_weighted/4FGL-DR3_6class_GMM_nmin100_prob_cat.csv


/var/folders/mb/yxqk032n4lvgsvnl3lqzbwvr0000gn/T/ipykernel_41413/1757752780.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_out['sample_weight'][~clf_mask] = 1. # put 1 for unassociated sources


In [26]:
df_out.columns

Index(['sin(GLAT)', 'cos(GLON)', 'sin(GLON)', 'log10(Energy_Flux100)',
       'log10(Unc_Energy_Flux100)', 'log10(Signif_Avg)', 'LP_index1000MeV',
       'LP_beta', 'LP_SigCurv', 'log10(Variability_Index)', 'CLASS1', '1_RF',
       '1_RF_std', '1_NN', '1_NN_std', '2_RF', '2_RF_std', '2_NN', '2_NN_std',
       '3_RF', '3_RF_std', '3_NN', '3_NN_std', '4_RF', '4_RF_std', '4_NN',
       '4_NN_std', '5_RF', '5_RF_std', '5_NN', '5_NN_std', '6_RF', '6_RF_std',
       '6_NN', '6_NN_std', 'sample_weight'],
      dtype='object')

In [27]:
sum(df_out['CLASS1'] == 'unk')

134

In [28]:
sample_ratio_train